## Import Prerequis

In [10]:
import sys
import os
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
from math import tanh
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Configuration des chemins vers la racine du projet
sys.path.append(os.path.abspath(os.path.join('..')))

### Compile Library C and generate JSON (make)

In [11]:
try:
    result = subprocess.run(
        "make -C ../libc clean && make -C ../libc",
        shell=True,
        capture_output=True,
        text=True
    )
    # print(result.stdout)

    if result.stderr:
        print(result.stderr)
    
    if result.returncode != 0:
        print(f"Build failed with exit code {result.returncode}")
        sys.exit(1)
    else:
        print("Build succeeded.")

except Exception as e:
    raise ValueError(f"Build failed: {e}")

Build succeeded.


## Import Loader

In [12]:
from engine.interop.loader import Loader
from engine.interop.linearModel import LinearModel

# Chargement du Singleton Loader
LIB_NAME = "libc"
LIB_FOLDER = "../libc"
BUILD_FOLDER = "../libc/build"
SPECS_FOLDER = "../libc/specs"
DEPENDENCIES_FOLDER = r"C:\msys64\mingw64\bin"

try:
    Loader.loadLibrary(LIB_NAME, lib_folder=LIB_FOLDER, build_folder=BUILD_FOLDER, specs_folder=SPECS_FOLDER, dependencies_bin_folder=DEPENDENCIES_FOLDER)
    print("✓ Bibliothèque C chargée avec succès !")
except Exception as e:
    if "already loaded" in str(e):
        print("✓ Bibliothèque C déjà chargée.")
    else:
        raise Exception(f"Erreur lors du chargement de la bibliothèque C : {e}")

✓ Bibliothèque C déjà chargée.


## Chargement du Dataset Réel (cleaned)

In [13]:
# Configuration globale du projet

config = {
    "dataset": {
        "csv_path": os.path.join("..", "dataset"),
        "data_folder_path": os.path.join("..", "dataset", "64x64"),
        "limit_per_category": 1000,
        "train_test_split_ratio": 0.7, # ex: 0.7 <=> 70% train, 30% test
    },
    "model": {
        "alpha": 0.001,
        "epochs": 1000,
    },
    "global": {
        "unknown_category": "unknown", # Nom réservé pour la catégorie inconnue (si le max n'appartient à aucune catégorie)
    }
}

In [14]:
# On définit les catégories et leurs chemins de données et CSV

categories = {
    "impressionism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "impressionism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "impressionism_clean.csv")
        },
    "realism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "realism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "realism_clean.csv")
    },
    "romanticism" : {
        "data_folder_path": os.path.join(config["dataset"]["data_folder_path"], "romanticism"),
        "csv_path": os.path.join(config["dataset"]["csv_path"], "romanticism_clean.csv")
    }
}

if config["global"]["unknown_category"] in categories:
    raise ValueError(f"Category name '{config['model']['unknown_category']}' is reserved and cannot be used.")

## Chargement du Dataset Réel (CSV)

In [15]:
# On charge les CSV et on les sépare en train / test pour chaque catégorie

df_csv_categories = dict()

df_csv_all_shuffled = {
    "train": pd.DataFrame(),
    "test": pd.DataFrame()
}

for category in categories:
    df = pd.read_csv(categories[category]["csv_path"])

    # Vérification si le DataFrame est vide
    if df.empty:
        raise ValueError(f"CSV file for category '{category}' is empty or not found at path: {categories[category]['csv_path']}")
    
    # Limiter le nombre d'images par catégorie si nécessaire
    if config["dataset"]["limit_per_category"] > 0:
        df = df.head(config["dataset"]["limit_per_category"])

    # Modifier le nom du fichier par son chemin complet
    if "Nom_Fichier" not in df.columns:
        raise ValueError("Column 'Nom_Fichier' does not exist in the DataFrame.")
    df["filepath"] = df["Nom_Fichier"].apply(lambda x: os.path.join(categories[category]["data_folder_path"], x))

    # Ajouter une colonne category (Y)
    for c in categories:
        if c in df.columns:
            raise ValueError(f"Column '{c}' already exists in the DataFrame.")
        df[c] = c == category
    
    # Separation test / train : % train, % test
    df_train = df.sample(frac=config["dataset"]["train_test_split_ratio"], random_state=42)
    df_test = df.drop(df_train.index)
    
    # On stocke les DataFrames train et test pour chaque catégorie
    df_csv_categories[category] = {
        "train": df_train,
        "test": df_test
    }

    # On concatène les DataFrames train et test pour toutes les catégories
    if df_csv_all_shuffled["train"].empty:
        df_csv_all_shuffled["train"] = df_train
        df_csv_all_shuffled["test"] = df_test
    else:
        df_csv_all_shuffled["train"] = pd.concat([df_csv_all_shuffled["train"], df_train], ignore_index=True)
        df_csv_all_shuffled["test"] = pd.concat([df_csv_all_shuffled["test"], df_test], ignore_index=True)

# On mélange les données pour éviter des conclusions via l'odre des catégories
df_csv_all_shuffled["train"] = df_csv_all_shuffled["train"].sample(frac=1, random_state=42).reset_index(drop=True)
df_csv_all_shuffled["test"] = df_csv_all_shuffled["test"].sample(frac=1, random_state=42).reset_index(drop=True)

In [16]:
# On prépare les données pour l'entraînement et le test

df_X_filepaths = {
    "train": df_csv_all_shuffled["train"]["filepath"].tolist(),
    "test": df_csv_all_shuffled["test"]["filepath"].tolist()
}

df_Y = {
    "train": dict(),
    "test": dict()
}

for category in categories:
    df_Y["train"][category] = list(df_csv_all_shuffled["train"][category])
    df_Y["test"][category] = list(df_csv_all_shuffled["test"][category])
    df_csv_all_shuffled["train"].drop(columns=[category], inplace=True)
    df_csv_all_shuffled["test"].drop(columns=[category], inplace=True)

## Chargement du Dataset Réel (Images NxN px) via Pillow

In [17]:
# On charge les images et on les normalise pour les mettre dans df_X

df_X = dict()

for step in df_X_filepaths:

    df_X[step] = []

    filepaths = df_X_filepaths[step]
    total = len(filepaths)

    for i, filepath in enumerate(filepaths):

        if i % 50 == 0 or i == total - 1:
            print(
                f"\rLoading {step}... {i+1}/{total} ({100*(i+1)/total:.1f}%)",
                end="",
                flush=True
            )

        img = Image.open(filepath).convert("RGB")

        # Normalisation de l'image (min / max) -> A CHANGER : normalisation par Standardisation
        img_array = (np.array(img).flatten() / 255.0).astype(np.float32)

        df_X[step].append(img_array)

    print()

df_X["train"] = np.concatenate(df_X["train"]).tolist()

Loading train... 2100/2100 (100.0%)
Loading test... 900/900 (100.0%)


## Train

In [ ]:
import tensorflow as tf
import datetime
import os

models = dict()
history = dict()

# 1. Configuration du dossier TensorBoard avec la date et l'heure
current_time = datetime.datetime.now().strftime("Art %d-%H:%M:%S")
log_dir = os.path.join("logs", "artgenre_linear", current_time)

# 2. Ecriture sur tensorboard
summary_writer = tf.summary.create_file_writer(log_dir)

for category in categories:
    print(f"Training model for category: {category}...")

    W_length = (len(df_X["train"]) // len(df_Y["train"][category]))
    models[category] = LinearModel.init_random(input_dim=W_length)
    
    # Exécution du code C
    history[category] = models[category].train(
        dataset_inputs=df_X["train"],
        dataset_expected_outputs=df_Y["train"][category],
        is_classification=True,
        alpha=config["model"]["alpha"], 
        epochs=config["model"]["epochs"]
    )
    print(f"Model for category '{category}' trained successfully.")

    # On indique à TensorFlow qu'on va écrire dans notre dossier
    with summary_writer.as_default():
        # On parcourt le tableau d'historique renvoyé par le C
        for epoch, loss_value in enumerate(history[category]):
            tf.summary.scalar(f"Loss/{category}", loss_value, step=epoch)
            
summary_writer.flush()

print(f"\nEntraînement terminé ! Les logs sont sauvegardés dans : {log_dir}")

I0000 00:00:1782920500.630264  121883 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782920500.679149  121883 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782920503.398583  121883 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1782920504.430627  121883 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1782920504.431251  122380 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERRO

Training model for category: impressionism...


## Test

In [ ]:
# On évalue les modèles sur le jeu de test

predictions = dict()

for category in categories:
    print(f"Evaluating model for category: {category}")
    
    if category in predictions:
        raise ValueError(f"Predictions for category '{category}' already exist.")
    
    predictions[category] = dict()
    predictions[category]["values"] = [models[category].predict(x, is_classification=False) for x in df_X["test"]]
    predictions[category]["prediction"] = [tanh(value) >= 0 for value in predictions[category]["values"]]

In [ ]:
# On détermine la catégorie prédite pour chaque image du jeu de test

df_predictions_test = list()

for i in range(len(predictions[list(categories.keys())[0]]["prediction"])):
    
    category_predicted = max(categories, key=lambda c: predictions[c]["values"][i])

    if predictions[category_predicted]["prediction"][i]:
        df_predictions_test.append(category_predicted)
    else:
        df_predictions_test.append(config["global"]["unknown_category"])

In [ ]:
# On détermine la catégorie attendue pour chaque image du jeu de test
df_predictions_expected = list()

for i in range(len(df_Y["test"][list(categories.keys())[0]])):
    category_expected = next((c for c in categories if df_Y["test"][c][i]), None)
    df_predictions_expected.append(category_expected)

In [ ]:
# Affichage de la matrice de confusion

ConfusionMatrixDisplay.from_predictions(
    df_predictions_expected,
    df_predictions_test,
    labels=list(categories.keys()) + [config["global"]["unknown_category"]],
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Confusion Matrix")
plt.show()